# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [ ]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [11]:
import os
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve()))

# Verify lib folder exists
if not Path("lib").exists():
    print("WARNING: 'lib' folder not found in", Path().resolve())
    print("Contents:", os.listdir("."))
    
# Create __init__.py if it doesn't exist
init_file = Path("lib/__init__.py")
init_file.touch()
print("Created:", init_file.resolve())

import json
import chromadb
from chromadb.utils import embedding_functions
from lib.vector_db import VectorStoreManager, CorpusLoaderService
from dotenv import load_dotenv
from openai import OpenAI



Created: /Users/jason.r.browning/Anthropic Training/lib/__init__.py


In [12]:
# TODO: Create a .env file with the following variables
# OPENAI_API_KEY="YOUR_KEY"
# CHROMA_OPENAI_API_KEY="YOUR_KEY"
# TAVILY_API_KEY="YOUR_KEY"

In [13]:
# TODO: Load environment variables
load_dotenv()

True

### VectorDB Instance

In [16]:
# TODO: Instantiate your ChromaDB Client
# Choose any path you want
# chroma_client = chromadb.PersistentClient(path="chromadb")

# Add project root to path so we can import src/
sys.path.insert(0, str(Path("../").resolve()))

openai_client = os.getenv("CHROMA_OPENAI_API_KEY")
chroma_client = chromadb.PersistentClient(path="./chromadb")


db = VectorStoreManager(os.getenv("CHROMA_OPENAI_API_KEY"))



loader_service = CorpusLoaderService(db)


### Collection

In [17]:
# TODO: Pick one embedding function
# If picking something different than openai, 
# make sure you use the same when loading it
# embedding_fn = embedding_functions.OpenAIEmbeddingFunction()

embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("OpenAI_API_KEY")
)


In [19]:
# TODO: Create a collection
# Choose any name you want
# collection = chroma_client.create_collection(
#    name="udaplay",
#    embedding_function=embedding_fn
#)

collection = chroma_client.create_collection(
    name="myudaplay2",
    embedding_function=embedding_fn
)

### Add documents

In [20]:
# Make sure you have a directory "project/starter/games"
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # You can change what text you want to index
    content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"

    # Use file name (like 001) as ID
    doc_id = os.path.splitext(file_name)[0]

    collection.add(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )